### Business Statement: Build the Regression model that  predict the total delivery duration in seconds from order creation to delivery completion.

### The objective of this project is to develop a machine learning model that accurately predicts the total delivery duration (in seconds) from order creation to delivery completion. The model uses historical order, store, and market data to identify patterns affecting delivery time.

### Step1: Data Gathering

In [1]:
from warnings import filterwarnings
filterwarnings('ignore')

In [2]:
path=r"C:\machine learning\repository\Dataset for regression.csv"
import pandas as pd
import numpy as np
df=pd.read_csv(path)
df.head()

,market_id,created_at,actual_delivery_time,store_id,store_primary_category,order_protocol,total_items,subtotal,num_distinct_items,min_item_price,max_item_price,total_onshift_dashers,total_busy_dashers,total_outstanding_orders,estimated_order_place_duration,estimated_store_to_consumer_driving_duration
0,1.0,22:24:17,23:27:16,1845,american,1.0,4,3441,4,557,1239,33.0,14.0,21.0,446,861.0
1,2.0,21:49:25,22:56:29,5477,mexican,2.0,1,1900,1,1400,1400,1.0,2.0,2.0,446,690.0
2,3.0,20:39:28,21:09:09,5477,NaN,1.0,1,1900,1,1900,1900,1.0,0.0,0.0,446,690.0
3,3.0,21:21:45,22:13:00,5477,NaN,1.0,6,6900,5,600,1800,1.0,1.0,2.0,446,289.0
4,3.0,02:40:36,03:20:26,5477,NaN,1.0,3,3900,3,1100,1600,6.0,6.0,9.0,446,650.0


### Step2: Perform basic data quality checks

In [3]:
df.shape

(197428, 16)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 197428 entries, 0 to 197427
Data columns (total 16 columns):
 #   Column                                        Non-Null Count   Dtype  
---  ------                                        --------------   -----  
 0   market_id                                     196441 non-null  float64
 1   created_at                                    197428 non-null  str    
 2   actual_delivery_time                          197421 non-null  str    
 3   store_id                                      197428 non-null  int64  
 4   store_primary_category                        192668 non-null  str    
 5   order_protocol                                196433 non-null  float64
 6   total_items                                   197428 non-null  int64  
 7   subtotal                                      197428 non-null  int64  
 8   num_distinct_items                            197428 non-null  int64  
 9   min_item_price                                197428 non-nu

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df.isna().sum()

market_id                                         987
created_at                                          0
actual_delivery_time                                7
store_id                                            0
store_primary_category                           4760
order_protocol                                    995
total_items                                         0
subtotal                                            0
num_distinct_items                                  0
min_item_price                                      0
max_item_price                                      0
total_onshift_dashers                           16262
total_busy_dashers                              16262
total_outstanding_orders                        16262
estimated_order_place_duration                      0
estimated_store_to_consumer_driving_duration      526
dtype: int64

In [7]:
for col in df.columns:
    if df[col].dtype == 'str':
        df[col].fillna(df[col].mode()[0], inplace=True)
    else:
        df[col].fillna(df[col].median(), inplace=True)

In [8]:
df['created_at'] = pd.to_datetime(df['created_at'],errors='coerce')
df['actual_delivery_time'] = pd.to_datetime(df['actual_delivery_time'],errors='coerce')  #It converts (string) → datetime format


In [9]:
df = df.dropna(subset=['actual_delivery_time'])   #It removes rows where delivery_time is missing (NaN)

In [10]:
df['delivery_time'] = (df['actual_delivery_time'] - df['created_at']).dt.total_seconds()   #This gives time difference.Converts that time difference into seconds
 #Target variable

In [11]:
df['order_hour'] = df['created_at'].dt.hour  #Extracts hour of the day
df['order_day'] = df['created_at'].dt.dayofweek

In [12]:
df = df.drop(['created_at', 'actual_delivery_time'], axis=1)

In [13]:
df = pd.get_dummies(df, drop_first=True)  #Converts categorical (text) columns into numeric columns (0/1)

### Separate X and Y features

In [14]:

X = df.drop('delivery_time', axis=1)  #Input variable
Y = df['delivery_time']     #Target variable

In [15]:
X.head()

,market_id,store_id,order_protocol,total_items,subtotal,num_distinct_items,min_item_price,max_item_price,total_onshift_dashers,total_busy_dashers,...,store_primary_category_southern,store_primary_category_spanish,store_primary_category_steak,store_primary_category_sushi,store_primary_category_tapas,store_primary_category_thai,store_primary_category_turkish,store_primary_category_vegan,store_primary_category_vegetarian,store_primary_category_vietnamese
0,1.0,1845,1.0,4,3441,4,557,1239,33.0,14.0,...,False,False,False,False,False,False,False,False,False,False
1,2.0,5477,2.0,1,1900,1,1400,1400,1.0,2.0,...,False,False,False,False,False,False,False,False,False,False
2,3.0,5477,1.0,1,1900,1,1900,1900,1.0,0.0,...,False,False,False,False,False,False,False,False,False,False
3,3.0,5477,1.0,6,6900,5,600,1800,1.0,1.0,...,False,False,False,False,False,False,False,False,False,False
4,3.0,5477,1.0,3,3900,3,1100,1600,6.0,6.0,...,False,False,False,False,False,False,False,False,False,False


In [16]:
Y.head()

0    3779.0
1    4024.0
2    1781.0
3    3075.0
4    2390.0
Name: delivery_time, dtype: float64

## Split the data into training and testing

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score



xtrain, xtest, ytrain, ytest = train_test_split(
    X, Y, train_size=0.80,test_size=0.2, random_state=42
)

In [18]:
print(f"Shape of training data: {xtrain.shape}")
print(f"Shape of testing data: {xtest.shape}")

Shape of training data: (157936, 88)
Shape of testing data: (39485, 88)


In [19]:
xtrain.head()

,market_id,store_id,order_protocol,total_items,subtotal,num_distinct_items,min_item_price,max_item_price,total_onshift_dashers,total_busy_dashers,...,store_primary_category_southern,store_primary_category_spanish,store_primary_category_steak,store_primary_category_sushi,store_primary_category_tapas,store_primary_category_thai,store_primary_category_turkish,store_primary_category_vegan,store_primary_category_vegetarian,store_primary_category_vietnamese
70519,3.0,416,4.0,11,3524,6,194,799,2.0,2.0,...,False,False,False,False,False,False,False,False,False,False
50630,1.0,4950,1.0,3,3500,2,1150,1200,59.0,60.0,...,False,False,False,False,False,False,False,False,False,False
57321,6.0,6845,2.0,2,1200,2,500,700,NaN,NaN,...,False,False,False,False,False,False,False,False,False,False
52856,2.0,2944,3.0,3,2675,3,775,1075,59.0,57.0,...,False,False,False,False,False,False,False,False,False,False
35924,6.0,2627,1.0,3,3200,2,300,750,NaN,NaN,...,True,False,False,False,False,False,False,False,False,False


In [20]:
xtest.head()

,market_id,store_id,order_protocol,total_items,subtotal,num_distinct_items,min_item_price,max_item_price,total_onshift_dashers,total_busy_dashers,...,store_primary_category_southern,store_primary_category_spanish,store_primary_category_steak,store_primary_category_sushi,store_primary_category_tapas,store_primary_category_thai,store_primary_category_turkish,store_primary_category_vegan,store_primary_category_vegetarian,store_primary_category_vietnamese
188418,2.0,5400,4.0,1,500,1,500,500,49.0,66.0,...,False,False,False,False,False,False,False,False,False,False
29851,3.0,6694,3.0,4,3785,3,549,1099,NaN,NaN,...,False,False,False,False,False,False,False,False,False,False
194087,2.0,4685,3.0,2,1340,2,400,535,6.0,7.0,...,False,False,False,False,False,False,False,False,False,False
82297,2.0,355,4.0,5,3097,3,249,695,76.0,50.0,...,False,False,False,False,False,False,False,False,False,False
52753,3.0,1758,5.0,1,1725,1,1725,1725,34.0,37.0,...,False,False,False,False,False,False,False,False,False,False


In [21]:
ytrain.head()

70519    4172.0
50630    4948.0
57321    3971.0
52856    4544.0
35924    4210.0
Name: delivery_time, dtype: float64

In [22]:
ytest.head()

188418    1913.0
29851     3145.0
194087    2031.0
82297     2208.0
52753     2089.0
Name: delivery_time, dtype: float64

### Data cleaning and data preprocessing

In [23]:
xtrain.info()

<class 'pandas.DataFrame'>
Index: 157936 entries, 70519 to 121962
Data columns (total 88 columns):
 #   Column                                        Non-Null Count   Dtype  
---  ------                                        --------------   -----  
 0   market_id                                     157142 non-null  float64
 1   store_id                                      157936 non-null  int64  
 2   order_protocol                                157157 non-null  float64
 3   total_items                                   157936 non-null  int64  
 4   subtotal                                      157936 non-null  int64  
 5   num_distinct_items                            157936 non-null  int64  
 6   min_item_price                                157936 non-null  int64  
 7   max_item_price                                157936 non-null  int64  
 8   total_onshift_dashers                         144869 non-null  float64
 9   total_busy_dashers                            144869 non-nul

In [24]:
xtrain.select_dtypes(include='object').columns

Index([], dtype='str')

In [25]:
list(xtrain.select_dtypes(include='str').columns)

[]

In [26]:
print(xtrain.dtypes.value_counts())

bool       73
int64       7
float64     6
int32       2
Name: count, dtype: int64


In [27]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import  StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

cat=list(xtrain.select_dtypes(include='str').columns)
con=list(xtrain.select_dtypes(include='number').columns)

con_pipe=make_pipeline(
    SimpleImputer(strategy='mean'),
    StandardScaler())

cat_pipe=make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown='ignore',sparse_output=False))

pre=ColumnTransformer([
    ("cat",cat_pipe,cat),
    ("con",con_pipe,con)
]).set_output(transform='pandas')  #output will be a Pandas DataFrame

In [28]:
pre

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('con', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

In [29]:
pre.fit(xtrain)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('con', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

In [30]:
xtrain_pre=pre.transform(xtrain)
xtest_pre=pre.transform(xtest)

In [31]:
xtrain_pre.head()

,con__market_id,con__store_id,con__order_protocol,con__total_items,con__subtotal,con__num_distinct_items,con__min_item_price,con__max_item_price,con__total_onshift_dashers,con__total_busy_dashers,con__total_outstanding_orders,con__estimated_order_place_duration,con__estimated_store_to_consumer_driving_duration,con__order_hour,con__order_day
70519,0.015048,-1.516165,0.743303,2.899956,0.462353,2.045805,-0.944330,-0.647163,-1.294686,-1.291658e+00,-1.111954e+00,-0.637677,-0.494015,1.217823,0.0
50630,-1.298991,0.691085,-1.257290,-0.070617,0.449196,-0.409335,0.885077,0.071659,0.429144,5.937596e-01,8.725049e-01,1.529228,1.217054,-0.746790,0.0
57321,1.986105,1.613612,-0.590426,-0.441939,-0.811711,-0.409335,-0.358767,-0.824628,0.000000,-2.309775e-16,-1.410043e-16,-0.637677,1.002600,1.448954,0.0
52856,-0.641972,-0.285480,0.076439,-0.070617,-0.003086,0.204450,0.167475,-0.152413,0.429144,4.962380e-01,1.051106e+00,-0.637677,-0.927486,-0.631224,0.0
35924,1.986105,-0.439802,-1.257290,-0.070617,0.284730,-0.409335,-0.741488,-0.735000,0.000000,-2.309775e-16,-1.410043e-16,1.529228,0.176724,1.102258,0.0


In [32]:
xtest_pre.head()

,con__market_id,con__store_id,con__order_protocol,con__total_items,con__subtotal,con__num_distinct_items,con__min_item_price,con__max_item_price,con__total_onshift_dashers,con__total_busy_dashers,con__total_outstanding_orders,con__estimated_order_place_duration,con__estimated_store_to_consumer_driving_duration,con__order_hour,con__order_day
188418,-0.641972,0.910154,0.743303,-0.813261,-1.195465,-1.023120,-0.358767,-1.183143,0.126717,7.888028e-01,-9.987991e-02,-0.637677,-1.160191,-0.631224,0.0
29851,0.015048,1.540102,0.076439,0.300704,0.605439,0.204450,-0.265000,-0.109391,0.000000,-2.309775e-16,-1.410043e-16,-0.637677,1.550143,-0.746790,0.0
194087,-0.641972,0.562077,0.076439,-0.441939,-0.734960,-0.409335,-0.550127,-1.120403,-1.173716,-1.129122e+00,-9.730417e-01,-0.637677,0.199539,0.871127,0.0
82297,-0.641972,-1.545861,0.743303,0.672026,0.228263,0.204450,-0.839082,-0.833591,0.943268,2.686876e-01,-1.395691e-01,-0.637677,0.683201,-0.977921,0.0
52753,0.015048,-0.862850,1.410168,-0.813261,-0.523895,-1.023120,1.985400,1.012760,-0.326922,-1.539059e-01,-3.380150e-01,-0.637677,-0.133550,-0.631224,0.0


In [33]:
mask = ytrain.notna()
xtrain_pre = xtrain_pre[mask]
ytrain = ytrain[mask]
mask_test = ytest.notna()

xtest_pre = xtest_pre[mask_test]
ytest = ytest[mask_test]

### Model Buidling

In [40]:
from sklearn.tree import DecisionTreeRegressor
model= DecisionTreeRegressor(
    criterion='squared_error',
    max_depth = 5,
    min_samples_split = 5,
    min_samples_leaf=10
)
model.fit(xtrain_pre,ytrain)

,"criterion criterion: {""squared_error"", ""friedman_mse"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in the half mean Poisson deviance to find splits... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"max_l

In [41]:
print(ytrain.isna().sum())

0


In [42]:
model.score(xtrain_pre,ytrain)

0.6762955125752845

In [43]:
model.score(xtest_pre,ytest)

0.6665002299959528

In [44]:
print("X test NaN:", pd.DataFrame(xtest_pre).isna().sum().sum())
print("y test NaN:", ytest.isna().sum())

X test NaN: 0
y test NaN: 0


### Hyperparameter tuning


In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV

# Define the parameter grid
params = { 
    'criterion': ['squared_error', 'absolute_error'], 
    'max_depth': [5, 10,15,20,None], 
    'min_samples_leaf': [2, 4, 6], 
    'min_samples_split': [2, 5, 10] 
}

base_model = DecisionTreeRegressor()


gscv = GridSearchCV(
    estimator=base_model,
    param_grid=params,
    cv=3,
    scoring='r2',
    n_jobs=-1 
)

# Fit the model
gscv.fit(xtrain_pre, ytrain)

# Access best parameters
print(f"Best Parameters: {gscv.best_params_}")

Best Parameters: {'criterion': 'squared_error', 'max_depth': 5, 'min_samples_leaf': 6, 'min_samples_split': 2}


In [ ]:
gscv.best_score_

np.float64(0.6621886438404526)

In [ ]:
best_dtr=gscv.best_estimator_

In [46]:
### Model Evaluation

In [ ]:
best_dtr.score(xtrain_pre,ytrain)

0.6765657481869222

In [ ]:
best_dtr.score(xtest_pre,ytest)

0.665876685484238

In [ ]:
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score


In [ ]:
ypreds=best_dtr.predict(xtest_pre)
ypreds=ypreds.round(2)
ypreds[:10]

array([3008.31, 3430.17, 2771.4 , 3053.28, 3008.31, 3053.28, 3008.31,
       2595.2 , 2390.7 , 2595.2 ])

In [ ]:
mse = mean_squared_error(ytest,ypreds)
rmse = mse**(1/2)
mae = mean_absolute_error(ytest,ypreds)
r2 = r2_score(ytest,ypreds)

In [ ]:
print(f"MSE:{mse:.2f}")
print(f"RMSE:{rmse:.2f}")
print(f"MAE:{mae:.2f}")
print(f"r2:{r2*100:.2f}%")

MSE:75322721.09
RMSE:8678.87
MAE:2399.97
r2:66.59%


In [47]:
mae_minutes = 2322 / 60
rmse_minutes = 8352 / 60

print(mae_minutes, rmse_minutes)

38.7 139.2


### Adjusted r2 score

In [ ]:
n = xtrain_pre.shape[0]
p = xtrain_pre.shape[1]
r2_train = r2_score(ytrain,model.predict(xtrain_pre))
adjusted_r2_train = 1 - (((1-r2_train)*(n-1))/(n-p-1))
round(adjusted_r2_train*100,2)

67.63

In [ ]:
n = xtest_pre.shape[0]
p = xtest_pre.shape[1]
adjusted_r2_test = 1 - (((1-r2)*(n-1))/(n-p-1))
round(adjusted_r2_test*100,2)

66.57

### We can finalise this model for out of sample predictions/deployment

In [ ]:
import joblib
joblib.dump(best_dtr,"model_dtr.joblib")
joblib.dump(con_pipe,"project_Regression.joblib")

['project_Regression.joblib']